# Install Requirements

In [1]:
!pip install selenium webdriver-manager dateparser tqdm requests

  Using cached requests-2.32.5-py3-none-any.whl.metadata (4.9 kB)
  Using cached charset_normalizer-3.4.3-cp311-cp311-win_amd64.whl.metadata (37 kB)
Using cached requests-2.32.5-py3-none-any.whl (64 kB)
Using cached charset_normalizer-3.4.3-cp311-cp311-win_amd64.whl (107 kB)



[notice] A new release of pip is available: 24.2 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


# Scraping

In [ ]:
import csv
import json
import logging
import random
import re
import sys
import threading
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime
from queue import Queue
from typing import Any

import dateparser
import requests
from selenium import webdriver
from selenium.common.exceptions import JavascriptException, StaleElementReferenceException, TimeoutException, WebDriverException
from selenium.webdriver.common.by import By
from selenium.webdriver.edge.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.microsoft import EdgeChromiumDriverManager
from tqdm import tqdm

# CONFIG
CONFIG = {
    "input_csv_path": "batch4.csv",
    "input_url_col_index": 4,
    "output_csv_path": "olx_housing_dataset_improved.csv",
    "raw_json_dir": None,   # set folder path like 'raw_appjson' to save raw window.__APP JSON for auditing, or None to disable
    "log_file": "olx_scraper_improved.log",
    "max_threads": 8,              # number of concurrent workers (controls ThreadPool)
    "max_drivers": 4,              # concurrent webdriver instances (Semaphore)
    "max_retries": 3,
    "wait_window_app": 10,         # seconds to wait for window.__APP
    "headless": True,
    "respectful_delay_min": 1.0,
    "respectful_delay_max": 2.5,
    "timeout_page_load": 30,
    "save_csv_streaming": True,    # streaming writer
    "user_agents": [
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:122.0) Gecko/20100101 Firefox/122.0",
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Edge/120.0.0.0",
    ],
    "robots_root": "https://www.olx.co.id",
    "log_level": logging.INFO
}

# Logging setup
logging.basicConfig(
    level=CONFIG["log_level"],
    format="%(asctime)s %(levelname)s:%(name)s: %(message)s",
    handlers=[logging.StreamHandler(sys.stdout), logging.FileHandler(CONFIG["log_file"])]
)
logger = logging.getLogger("olx-scraper-improved")

# Globals
DRIVER_SEMAPHORE = threading.Semaphore(CONFIG["max_drivers"])
WRITE_QUEUE: "Queue[dict]" = Queue()

# Utilities
def respectful_pause():
    time.sleep(random.uniform(CONFIG["respectful_delay_min"], CONFIG["respectful_delay_max"]))

def check_robots() -> bool:
    """
    Basic robots.txt check. Returns False if it appears disallowed (heuristic).
    This is a conservative check — always manually inspect robots.txt and ToS.
    """
    url = CONFIG["robots_root"].rstrip("/") + "/robots.txt"
    try:
        r = requests.get(url, timeout=6)
        if r.status_code == 200:
            text = r.text.lower()
            if "disallow" in text and "/item/" in text:
                logger.warning("robots.txt may disallow '/item/' scraping. Please inspect: %s", url)
                return False
            return True
        else:
            return True
    except Exception as e:
        logger.warning("Couldn't fetch robots.txt: %s", e)
        return True

def pick_user_agent() -> str:
    return random.choice(CONFIG["user_agents"])

def make_driver(user_agent: str | None = None) -> webdriver.Edge:
    """
    Create Edge WebDriver using webdriver-manager. Controlled by DRIVER_SEMAPHORE.
    Caller MUST call teardown_driver(driver) when done.
    """
    DRIVER_SEMAPHORE.acquire()
    try:
        options = Options()
        if CONFIG["headless"]:
            # new headless mode flag for modern browsers
            options.add_argument("--headless=new")
        options.add_argument("--inprivate")
        options.add_argument("--disable-gpu")
        options.add_argument("--no-sandbox")
        options.add_argument("--disable-logging")
        if user_agent:
            options.add_argument(f"--user-agent={user_agent}")
        driver_path = EdgeChromiumDriverManager().install()
        driver = webdriver.Edge(driver_path, options=options)
        driver.set_page_load_timeout(CONFIG["timeout_page_load"])
        return driver
    except Exception as e:
        DRIVER_SEMAPHORE.release()
        raise

def teardown_driver(driver: webdriver.Edge):
    try:
        driver.quit()
    except Exception as e:
        logger.debug("Error quitting driver: %s", e)
    finally:
        DRIVER_SEMAPHORE.release()

def wait_for_window_app(driver: webdriver.Edge, timeout: int = None) -> Any:
    """
    Explicitly wait until window.__APP exists (and return it).
    Returns None on timeout/failure.
    """
    t = timeout or CONFIG["wait_window_app"]
    try:
        def _check(drv):
            try:
                val = drv.execute_script("return (typeof window !== 'undefined' && window.__APP) ? window.__APP : null;")
                return val
            except JavascriptException:
                return False
        return WebDriverWait(driver, t).until(_check)
    except TimeoutException:
        return None

def extract_id_from_url(url: str) -> str | None:
    m = re.search(r'-iid-(\d+)$', url)
    return m.group(1) if m else None

def parse_posting_date(raw: str):
    if not raw:
        return None, None, None
    try:
        dt = dateparser.parse(raw)
        if dt:
            return dt.strftime("%d %B %Y"), dt.month, dt.year
    except Exception:
        pass
    return None, None, None

def get_nested(data: dict, keys: list, default=None):
    dd = data
    for k in keys:
        if isinstance(dd, list):
            try:
                dd = dd[k]
            except (IndexError, TypeError):
                return default
        else:
            if not isinstance(dd, dict):
                return default
            dd = dd.get(k, default)
            if dd is default:
                return default
    return dd or default

# Description parsing utilities
def split_sentences(text: str) -> list:
    if not text:
        return []
    return [line.strip() for line in text.splitlines() if line.strip()]

def scrape_description_value(lines: list, keywords: list, entity_type: str | None = None):
    for k in keywords:
        if entity_type == "electricity":
            patt = rf'{re.escape(k)}\s*[\:\.\-\s]*([\d.,]+)\s*(watt|va|kva|token|w|wt|kwh)?'
        elif entity_type in ("garage", "carport"):
            patt = rf'{re.escape(k)}\s*[\:\.\-\s]*(\d+)?\s*(mobil|mbl|cars?)?'
        elif entity_type == "heading":
            patt = rf'{re.escape(k)}\s*[\:\.\-\s]*(.*?)(Timur|Tenggara|Selatan|Barat Daya|Barat Laut|Barat|Utara|Timur Laut|Kiblat|Khiblat)'
        else:
            patt = rf'{re.escape(k)}\s*[\:\.\-\s]*([\w\s\.,]+)'
        for s in lines:
            if k.lower() in s.lower():
                m = re.search(patt, s, re.IGNORECASE)
                if m:
                    if entity_type == "electricity":
                        val = m.group(1).replace(".", "").replace(",", "")
                        try:
                            return int(val)
                        except Exception:
                            return None
                    if entity_type in ("garage", "carport"):
                        v = m.group(1)
                        return int(v) if v else 1
                    if entity_type == "heading":
                        h = m.group(2).capitalize()
                        if h.lower() in ("kiblat", "khiblat"):
                            return "Barat"
                        return h
                    return m.group(1).strip()
    return None

def count_rooms(lines: list, keywords: list):
    for kw in keywords:
        patt = rf'(\d+)?\s*{re.escape(kw)}'
        for s in lines:
            m = re.search(patt, s, re.IGNORECASE)
            if m:
                return int(m.group(1)) if m.group(1) else 1
    return 0

def extract_description_details(lines: list) -> dict:
    keys = {
        "garage": ["Garasi", "Garage"],
        "carport": ["Carport", "Carpot"],
        "electricity": ["Listrik", "pln", "listtrik"],
        "heading": ["Hadap", "Orientasi"],
        "ruang_tamu": ["Ruang Tamu"],
        "ruang_makan": ["Ruang Makan"],
        "maid_bedroom": ["Kamar Pembantu"],
        "maid_bathroom": ["Kamar Mandi Pembantu"],
        "floors": ["Lantai"]
    }
    lines = lines or []
    info = {}
    info["garage"] = scrape_description_value(lines, keys["garage"], "garage") or 0
    info["carport"] = scrape_description_value(lines, keys["carport"], "carport") or 0
    info["electricity"] = scrape_description_value(lines, keys["electricity"], "electricity")
    info["heading"] = scrape_description_value(lines, keys["heading"], "heading")
    info["ruang_tamu"] = count_rooms(lines, keys["ruang_tamu"])
    info["ruang_makan"] = count_rooms(lines, keys["ruang_makan"])
    info["maid_bedroom"] = count_rooms(lines, keys["maid_bedroom"])
    info["maid_bathroom"] = count_rooms(lines, keys["maid_bathroom"])
    # floors
    floor_val = None
    for s in lines:
        m = re.search(r'(\d+)\s*lantai|lantai\s*[\:\-\s]*(\d+)', s, re.IGNORECASE)
        if m:
            floor_val = m.group(1) or m.group(2)
            break
    info["floors"] = int(floor_val) if floor_val else 1
    other_rooms = [s for s in lines if "ruang" in s.lower() and "ruang tamu" not in s.lower() and "ruang makan" not in s.lower()]
    info["additional_rooms"] = 1 if other_rooms else 0
    return info

# Extract window.__APP robustly
def extract_app_from_page(driver: webdriver.Edge) -> Any:
    """
    Attempt explicit retrieval of window.__APP. Fallback to scanning & executing <script> tags that contain it.
    Returns the app_data dict/object or None on failure.
    """
    app = wait_for_window_app(driver, timeout=CONFIG["wait_window_app"])
    if app:
        return app

    # fallback: scan script tags
    try:
        scripts = driver.find_elements(By.TAG_NAME, "script")
        for s in scripts:
            try:
                txt = s.get_attribute("innerHTML")
                if txt and "window.__APP" in txt:
                    # execute the script content (same as original code)
                    try:
                        driver.execute_script(txt)
                        app = wait_for_window_app(driver, timeout=5)
                        if app:
                            return app
                    except JavascriptException:
                        continue
            except StaleElementReferenceException:
                continue
    except Exception:
        pass
    return None

# Single URL scraping
def scrape_one_url(url: str) -> dict | None:
    """
    Scrape one OLX listing URL. Implements retries, backoff, random UA, respectful delays.
    """
    if not check_robots():
        logger.error("Robots check failed — aborting scraping per policy. (%s)", CONFIG["robots_root"])
        return None

    ua = pick_user_agent()
    for attempt in range(1, CONFIG["max_retries"] + 1):
        driver = None
        try:
            driver = make_driver(user_agent=ua)
            driver.get(url)
            respectful_pause()

            app = extract_app_from_page(driver)
            if not app:
                raise RuntimeError("window.__APP not found")

            ads_id = extract_id_from_url(url) or ""
            # fields
            title = get_nested(app, ["states", "items", "elements", ads_id, "title"])
            price = get_nested(app, ["states", "items", "elements", ads_id, "price", "value", "raw"])
            desc_raw = get_nested(app, ["states", "items", "elements", ads_id, "description"]) or ""
            desc_lines = split_sentences(desc_raw)
            desc_detail = extract_description_details(desc_lines)

            # seller & images
            seller_id = get_nested(app, ["states", "items", "elements", ads_id, "user_id"])
            seller_name = get_nested(app, ["states", "users", "elements", str(seller_id), "name"]) if seller_id else None
            images = []
            img_objs = get_nested(app, ["states", "items", "elements", ads_id, "images"], [])
            if isinstance(img_objs, list):
                for im in img_objs:
                    if isinstance(im, dict) and im.get("url"):
                        images.append(im.get("url"))

            # parameters lookup function
            params = get_nested(app, ["states", "items", "elements", ads_id, "parameters"], []) or []
            def param_value(key_name):
                for p in params:
                    if p.get("key_name", "").lower() == key_name.lower():
                        if p.get("type") == "single":
                            return p.get("value_name")
                        if p.get("type") == "multi":
                            vals = [v.get("value_name") for v in p.get("values", [])]
                            return vals[0] if len(vals) == 1 else vals
                return None

            certificate_raw = param_value("Sertifikasi")
            certificate = None
            if certificate_raw:
                if isinstance(certificate_raw, str) and "shm" in certificate_raw.lower():
                    certificate = "SHM"
                else:
                    certificate = certificate_raw

            posting_raw = get_nested(app, ["states", "items", "elements", ads_id, "created_at"])
            posting_date, posting_month, posting_year = parse_posting_date(posting_raw)

            # assemble result
            result = {
                "url": url,
                "ads_id": ads_id,
                "title": title,
                "price": price,
                "type": param_value("Tipe"),
                "land_area": param_value("Luas Tanah"),
                "building_area": param_value("Luas Bangunan"),
                "bedrooms": param_value("Kamar Tidur"),
                "bathrooms": param_value("Kamar Mandi"),
                "certificate": certificate,
                "address": param_value("Alamat Lokasi"),
                "address_city": get_nested(app, ["states", "items", "elements", ads_id, "metadata", "locations", 0, "tree", 0, "addressComponents", 1, "name"]),
                "seller": seller_name,
                "images": images,
                "garage_capacity": desc_detail["garage"],
                "carport_capacity": desc_detail["carport"],
                "electricity_capacity": desc_detail["electricity"],
                "house_orientation": desc_detail["heading"],
                "maid_bedrooms": desc_detail["maid_bedroom"],
                "maid_bathrooms": desc_detail["maid_bathroom"],
                "ruang_tamu": desc_detail["ruang_tamu"],
                "ruang_makan": desc_detail["ruang_makan"],
                "additional_rooms": desc_detail["additional_rooms"],
                "floors": param_value("Lantai") or desc_detail["floors"],
                "facilities": param_value("Fasilitas"),
                "description": desc_lines,
                "posting_date": posting_date,
                "posting_month": posting_month,
                "posting_year": posting_year,
                "_scraped_at": datetime.utcnow().isoformat()
            }

            # optionally save raw app JSON for audit
            if CONFIG["raw_json_dir"]:
                try:
                    import os
                    os.makedirs(CONFIG["raw_json_dir"], exist_ok=True)
                    fname = f"{CONFIG['raw_json_dir']}/{result['ads_id'] or 'noid'}_{int(time.time())}.json"
                    with open(fname, "w", encoding="utf-8") as jf:
                        json.dump(app, jf, ensure_ascii=False)
                except Exception as e:
                    logger.debug("Failed saving raw app json: %s", e)

            logger.info("Scraped %s", url)
            return result

        except Exception as e:
            logger.warning("Attempt %d failed for %s: %s", attempt, url, e)
            backoff = (2 ** (attempt - 1)) + random.random()
            time.sleep(backoff)
            continue
        finally:
            if 'driver' in locals() and driver:
                try:
                    teardown_driver(driver)
                except Exception:
                    pass

    logger.error("All attempts failed for %s", url)
    return None

# Streaming CSV writer (thread)
def csv_writer_worker(fieldnames: list):
    with open(CONFIG["output_csv_path"], "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        while True:
            item = WRITE_QUEUE.get()
            if item is None:
                break
            # sanitize description list -> join
            if isinstance(item.get("description"), list):
                item["description"] = " ||| ".join(item["description"])
            writer.writerow(item)
            WRITE_QUEUE.task_done()

# Orchestration
def load_urls_from_csv(path: str, col_index: int) -> list:
    urls = []
    with open(path, newline="", encoding="utf-8") as f:
        r = csv.reader(f)
        for row in r:
            if not row:
                continue
            if len(row) > col_index:
                urls.append(row[col_index])
            else:
                urls.append(row[-1])
    return urls

def scrape_all(urls: list):
    """
    Parallel scraping using ThreadPoolExecutor.
    Each task creates its own webdriver (bounded by DRIVER_SEMAPHORE).
    Results are streamed to CSV via WRITE_QUEUE.
    """
    # prepare fieldnames - ensure consistent ordering
    fieldnames = [
        "url","ads_id","title","price","type","land_area","building_area","bedrooms","bathrooms",
        "certificate","address","address_city","seller","images",
        "garage_capacity","carport_capacity","electricity_capacity","house_orientation",
        "maid_bedrooms","maid_bathrooms","ruang_tamu","ruang_makan","additional_rooms","floors",
        "facilities","description","posting_date","posting_month","posting_year","_scraped_at"
    ]
    writer_thread = threading.Thread(target=csv_writer_worker, args=(fieldnames,), daemon=True)
    writer_thread.start()

    results = []
    with ThreadPoolExecutor(max_workers=CONFIG["max_threads"]) as ex:
        futures = {ex.submit(scrape_one_url, u): u for u in urls}
        for fut in tqdm(as_completed(futures), total=len(futures), desc="Scraping URLs"):
            url = futures[fut]
            try:
                res = fut.result()
                if res:
                    WRITE_QUEUE.put(res)
                    results.append(res)
            except Exception as e:
                logger.exception("Unhandled exception scraping %s: %s", url, e)

    # signal writer to finish
    WRITE_QUEUE.put(None)
    writer_thread.join()
    return results

# Main
def main():
    logger.info("Starting OLX scraping (combined) ...")
    try:
        urls = load_urls_from_csv(CONFIG["input_csv_path"], CONFIG["input_url_col_index"])
    except FileNotFoundError:
        logger.error("Input CSV not found: %s", CONFIG["input_csv_path"])
        return
    if not urls:
        logger.warning("No URLs found in input CSV.")
        return
    logger.info("URLs to scrape: %d", len(urls))
    start = time.time()
    results = scrape_all(urls)
    elapsed = time.time() - start
    logger.info("Scraping finished. Success: %d / %d. Elapsed: %.1fs", len(results), len(urls), elapsed)

if __name__ == "__main__":
    main()

## Gemini

In [ ]:
import csv
import logging
import re
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from dateparser import parse
from selenium import webdriver
from selenium.common.exceptions import (StaleElementReferenceException,
                                        TimeoutException)
from selenium.webdriver.common.by import By
from selenium.webdriver.edge.options import Options
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait
from tqdm import tqdm

# Configuration
CONFIG = {
    'input_csv_path': 'batch4.csv',
    'output_csv_path': 'olx_housing_dataset_final.csv',
    'log_file_path': 'scraper.log',
    'max_threads': 10,  # Dapat disesuaikan berdasarkan kemampuan komputer
    'url_column_index': 4,
    'timeout': 20 # Waktu tunggu maksimal untuk memuat halaman
}

# Logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - [%(threadName)s] - %(message)s',
    handlers=[
        logging.FileHandler(CONFIG['log_file_path']),
        logging.StreamHandler()
    ]
)

def extract_id_from_url(url):
    """Mengekstrak ID iklan dari URL menggunakan Regex."""
    pattern = r'-iid-(\d+)$'
    match = re.search(pattern, url)
    return match.group(1) if match else None

def get_nested_value(data_dict, keys, default=None):
    """Mengakses nilai dalam dictionary/list bersarang secara aman."""
    for key in keys:
        if isinstance(data_dict, list):
            try:
                data_dict = data_dict[key]
            except (IndexError, TypeError):
                return default
        else:
            data_dict = data_dict.get(key, default)
            if data_dict is default:
                return default
    return data_dict or default

def get_data(app_data, ads_ID, data_type, key_name=None, location_type=None):
    """Mengambil data spesifik dari objek __APP."""
    if data_type == "location" and location_type:
        address_components = get_nested_value(app_data, ["states", "items", "elements", ads_ID, "metadata", "locations", 0, "tree", 0, "addressComponents"], [])
        for component in address_components:
            if component.get("type") == location_type:
                return component.get("name")
        return None
    elif data_type == "images":
        images = get_nested_value(app_data, ["states", "items", "elements", ads_ID, "images"], [])
        return [image.get("url") for image in images if 'url' in image]
    elif data_type == "parameters" and key_name:
        parameters = get_nested_value(app_data, ["states", "items", "elements", ads_ID, "parameters"], [])
        for param in parameters:
            if param.get("key_name", "").lower() == key_name.lower():
                if param.get("type") == "single":
                    return param.get("value_name")
                elif param.get("type") == "multi":
                    values = [value.get("value_name") for value in param.get("values", [])]
                    return values[0] if len(values) == 1 else values
        return None
    return None

def get_seller_name(app_data, ads_ID):
    """Mendapatkan nama penjual dari objek __APP."""
    seller_id = get_nested_value(app_data, ["states", "items", "elements", ads_ID, "user_id"])
    return get_nested_value(app_data, ["states", "users", "elements", str(seller_id), "name"])

def extract_app_data(driver):
    """Mengekstrak objek JavaScript window.__APP dari halaman."""
    retries = 3
    for _ in range(retries):
        try:
            script_tags = driver.find_elements(By.TAG_NAME, 'script')
            for script in script_tags:
                script_content = script.get_attribute('innerHTML')
                if 'window.__APP' in script_content:
                    driver.execute_script(script_content)
                    return driver.execute_script("return window.__APP;")
        except StaleElementReferenceException:
            logging.warning("Stale element reference, mencoba lagi...")
            time.sleep(1)
            continue
    logging.error("Stale element, semua percobaan gagal.")
    return None

def split_sentences(description):
    """Memecah teks deskripsi menjadi beberapa kalimat."""
    sentences = description.splitlines()
    return [line.strip() for line in sentences if line.strip()]

def scrape_description(ads_description, kata_kunci_list, entity_type=None):
    """Fungsi umum untuk mengekstrak info dari deskripsi menggunakan Regex."""
    if not isinstance(ads_description, list): return None
    for kata_kunci in kata_kunci_list:
        if entity_type == 'electricity': pattern = rf'{re.escape(kata_kunci)}\s*[\:\.\-\s]*([\d.,]+)\s*(watt|va|kva|w|wt|kwh)?'
        elif entity_type in ['garage', 'carport']: pattern = rf'{re.escape(kata_kunci)}\s*[\:\.\-\s]*(\d+)?\s*(mobil|mbl)?'
        elif entity_type == 'heading': pattern = rf'{re.escape(kata_kunci)}\s*[\:\.\-\s]*(.*?)(Timur|Tenggara|Selatan|Barat Daya|Barat Laut|Barat|Utara|Kiblat|Khiblat)'
        else: pattern = rf'{re.escape(kata_kunci)}\s*[\:\.\-\s]*([\w\s\.,]+)'
        
        for sentence in ads_description:
            if isinstance(sentence, str) and kata_kunci.lower() in sentence.lower():
                match = re.search(pattern, sentence, re.IGNORECASE)
                if match:
                    if entity_type == 'electricity':
                        value_str = match.group(1).replace('.', '').replace(',', '')
                        return int(value_str)
                    elif entity_type in ['garage', 'carport']:
                        value_str = match.group(1)
                        return int(value_str) if value_str else 1
                    elif entity_type == 'heading':
                        heading = match.group(2).capitalize()
                        return 'Barat' if heading.lower() in ['kiblat', 'khiblat'] else heading
                    else:
                        return match.group(1).strip().split(" ")[0]
    return None

def count_rooms(ads_description, keyword_list):
    """Menghitung jumlah ruangan dari deskripsi."""
    for keyword in keyword_list:
        pattern = rf'(\d+)?\s*{re.escape(keyword)}'
        for sentence in ads_description:
            match = re.search(pattern, sentence, re.IGNORECASE)
            if match:
                return int(match.group(1)) if match.group(1) else 1
    return 0

def extract_floors(ads_description, keyword_list):
    """Mengekstrak jumlah lantai dari deskripsi."""
    for keyword in keyword_list:
        pattern = rf'(\d+)\s*{re.escape(keyword)}|{re.escape(keyword)}\s*[\:\-\s]*(\d+)'
        for sentence in ads_description:
            match = re.search(pattern, sentence, re.IGNORECASE)
            if match:
                if match.group(1): return int(match.group(1))
                alt_match = re.search(rf'{re.escape(keyword)}\s*[\:\-\s]*(\d+)', sentence, re.IGNORECASE)
                if alt_match: return int(alt_match.group(1))
    return 1 # Default 1 lantai jika tidak disebutkan

def parse_date(ads_postingdate):
    """Mengubah format tanggal menjadi lebih standar."""
    if not ads_postingdate: return None, None, None
    date_obj = parse(ads_postingdate, languages=['id', 'en'])
    if date_obj:
        return f"{date_obj.day} {date_obj.strftime('%B')} {date_obj.year}", date_obj.month, date_obj.year
    return None, None, None
    
def extract_all_ads_info(ads_description):
    """Mengekstrak semua informasi tambahan dari teks deskripsi."""
    if not isinstance(ads_description, list): ads_description = []
    keywords = {'garage': ['Garasi'],'carport': ['Carport'],'electricity': ['Listrik', 'pln'],'heading': ['Hadap'],'ruang_tamu': ['Ruang Tamu'],'ruang_makan': ['Ruang Makan'],'maid_bedroom': ['Kamar Pembantu'],'maid_bathroom': ['Kamar Mandi Pembantu'],'floors': ['Lantai']}
    info = {
        'garage': scrape_description(ads_description, keywords['garage'], 'garage') or 0,
        'carport': scrape_description(ads_description, keywords['carport'], 'carport') or 0,
        'electricity': scrape_description(ads_description, keywords['electricity'], 'electricity'),
        'heading': scrape_description(ads_description, keywords['heading'], 'heading'),
        'ruang_tamu': count_rooms(ads_description, keywords['ruang_tamu']),
        'ruang_makan': count_rooms(ads_description, keywords['ruang_makan']),
        'maid_bedroom': count_rooms(ads_description, keywords['maid_bedroom']),
        'maid_bathroom': count_rooms(ads_description, keywords['maid_bathroom']),
        'floors': extract_floors(ads_description, keywords['floors'])
    }
    other_rooms = sum(1 for s in ads_description if "ruang" in s.lower() and "ruang tamu" not in s.lower() and "ruang makan" not in s.lower())
    info['additional_rooms'] = 1 if other_rooms > 0 else 0
    return info

def create_driver():
    """Membuat dan mengembalikan instance Selenium WebDriver."""
    options = Options()
    options.add_argument("--inprivate")
    options.add_argument("--headless")
    options.add_argument("--disable-gpu")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-logging")
    options.add_argument("log-level=3")
    return webdriver.Edge(options=options)

# Menerima instance driver
def scrape_url(driver, url):
    """Melakukan scraping pada satu URL menggunakan instance driver yang sudah ada."""
    try:
        driver.get(url)
        WebDriverWait(driver, CONFIG['timeout']).until(
            EC.presence_of_element_located((By.TAG_NAME, "script"))
        )

        ads_ID = extract_id_from_url(url)
        if not ads_ID:
            logging.error(f"Gagal ekstrak Ad ID dari URL: {url}")
            return None

        app_data = extract_app_data(driver)
        if not app_data:
            logging.error(f"Gagal ekstrak data __APP dari URL: {url}")
            return None

        # Ekstraksi Data
        desc_raw = get_nested_value(app_data, ["states", "items", "elements", ads_ID, "description"])
        desc = split_sentences(desc_raw) if desc_raw else []
        info = extract_all_ads_info(desc)
        date_raw = get_nested_value(app_data, ["states", "items", "elements", ads_ID, "created_at"])
        date, month, year = parse_date(date_raw)
        
        scraped_data = {
            'url': url, 'ads_id': ads_ID,
            'title': get_nested_value(app_data, ["states", "items", "elements", ads_ID, "title"]),
            'price': get_nested_value(app_data, ["states", "items", "elements", ads_ID, "price", "value", "raw"]),
            'type': get_data(app_data, ads_ID, "parameters", key_name="Tipe"),
            'land_area': get_data(app_data, ads_ID, "parameters", key_name="Luas Tanah"),
            'building_area': get_data(app_data, ads_ID, "parameters", key_name="Luas Bangunan"),
            'bedrooms': get_data(app_data, ads_ID, "parameters", key_name="Kamar Tidur"),
            'bathrooms': get_data(app_data, ads_ID, "parameters", key_name="Kamar Mandi"),
            'maid_bedrooms': info.get('maid_bedroom'), 'maid_bathrooms': info.get('maid_bathroom'),
            'ruang_tamu': info.get('ruang_tamu'), 'ruang_makan': info.get('ruang_makan'),
            'additional_rooms': info.get('additional_rooms'),
            'floors': get_data(app_data, ads_ID, "parameters", key_name="Lantai") or info.get('floors'),
            'certificate': get_data(app_data, ads_ID, "parameters", key_name="Sertifikasi"),
            'address': get_data(app_data, ads_ID, "parameters", key_name="Alamat Lokasi"),
            'address_city': get_data(app_data, ads_ID, "location", location_type="CITY"),
            'garage_capacity': info.get('garage'), 'carport_capacity': info.get('carport'),
            'facilities': get_data(app_data, ads_ID, "parameters", key_name="Fasilitas"),
            'description': desc, 'posting_date': date,
            'posting_date_month': month, 'posting_date_year': year,
            'poster': get_seller_name(app_data, ads_ID),
            'electricity_capacity': info.get('electricity'), 'house_orientation': info.get('heading'),
            'image_url': get_data(app_data, ads_ID, "images")
        }
        logging.info(f"Berhasil scrape: {url}")
        return scraped_data

    except TimeoutException:
        logging.error(f"Timeout saat memuat URL: {url}")
        return None
    except Exception as e:
        logging.error(f"Eror tak terduga saat scrape {url}: {e}", exc_info=True)
        return None

def worker(url_list):
    """Fungsi worker untuk satu thread. Membuat satu driver dan memproses daftar URL."""
    driver = create_driver()
    results = []
    try:
        for url in url_list:
            data = scrape_url(driver, url)
            if data:
                results.append(data)
    finally:
        driver.quit()
    return results

def main():
    logging.info("Script scraper dimulai.")
    try:
        with open(CONFIG['input_csv_path'], mode='r', encoding='utf-8') as f:
            reader = csv.reader(f)
            listing_urls = [row[CONFIG['url_column_index']] for row in reader if row and len(row) > CONFIG['url_column_index']]
    except FileNotFoundError:
        logging.error(f"File input tidak ditemukan: {CONFIG['input_csv_path']}")
        return
    
    if not listing_urls:
        logging.warning("Tidak ada URL yang ditemukan di file input.")
        return

    logging.info(f"Ditemukan {len(listing_urls)} URL untuk di-scrape.")
    
    all_scraped_data = []
    # Membagi daftar URL untuk setiap thread
    chunk_size = (len(listing_urls) + CONFIG['max_threads'] - 1) // CONFIG['max_threads']
    url_chunks = [listing_urls[i:i + chunk_size] for i in range(0, len(listing_urls), chunk_size)]

    with ThreadPoolExecutor(max_workers=CONFIG['max_threads']) as executor:
        futures = [executor.submit(worker, chunk) for chunk in url_chunks]
        
        for future in tqdm(as_completed(futures), total=len(futures), desc="Scraping Progress"):
            try:
                result_chunk = future.result()
                if result_chunk:
                    all_scraped_data.extend(result_chunk)
            except Exception as e:
                logging.error(f"Sebuah thread worker gagal: {e}", exc_info=True)

    if not all_scraped_data:
        logging.warning("Tidak ada data yang berhasil di-scrape.")
        return

    # Menulis hasil ke file CSV
    try:
        with open(CONFIG['output_csv_path'], 'w', newline='', encoding='utf-8') as csvfile:
            # Menggunakan keys dari data pertama sebagai fieldnames
            fieldnames = all_scraped_data[0].keys()
            writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(all_scraped_data)
        logging.info(f"Berhasil membuat file CSV: {CONFIG['output_csv_path']}")
    except (IOError, IndexError) as e:
        logging.error(f"Gagal menulis ke file CSV: {e}")

if __name__ == "__main__":
    main()

## GPT

In [ ]:
import csv
import logging
import random
import re
import sys
import threading
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime
from queue import Queue

import requests
from dateutil import parser as dateparser
from selenium.common.exceptions import (
    JavascriptException,
    StaleElementReferenceException,
    TimeoutException,
)
from selenium.webdriver.edge.options import Options
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait
from selenium import webdriver
from selenium.webdriver.common.by import By
from webdriver_manager.microsoft import EdgeChromiumDriverManager
from tqdm import tqdm

# KONFIGURASI
MAX_THREADS = 8
MAX_RETRIES = 3
WAIT_WINDOW_APP = 10
CSV_OUTPUT = "olx_housing_dataset_final.csv"
USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:122.0) Gecko/20100101 Firefox/122.0",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Edge/120.0.0.0",
]
DRIVER_SEMAPHORE = threading.Semaphore(MAX_THREADS)
write_queue: "Queue[dict]" = Queue()

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s:%(name)s: %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)],
)
logger = logging.getLogger("olx-scraper")

# UTILITAS UMUM
def respectful_pause(min_sec=1.0, max_sec=2.5):
    time.sleep(random.uniform(min_sec, max_sec))

def check_robots(website_root: str) -> bool:
    robots_url = website_root.rstrip("/") + "/robots.txt"
    try:
        r = requests.get(robots_url, timeout=6)
        if r.status_code == 200:
            body = r.text.lower()
            if "disallow" in body and "/item/" in body:
                logger.warning("robots.txt may disallow /item/ scraping. Check %s", robots_url)
                return False
            return True
        return True
    except Exception as e:
        logger.warning("Could not fetch robots.txt: %s", e)
        return True

def setup_driver(user_agent=None, headless=True):
    DRIVER_SEMAPHORE.acquire()
    try:
        options = Options()
        if headless:
            options.add_argument("--headless=new")
        options.add_argument("--disable-gpu")
        options.add_argument("--no-sandbox")
        options.add_argument("--disable-logging")
        options.add_argument("--inprivate")
        if user_agent:
            options.add_argument(f"--user-agent={user_agent}")

        driver = webdriver.Edge(EdgeChromiumDriverManager().install(), options=options)
        driver.set_page_load_timeout(30)
        return driver
    except Exception:
        DRIVER_SEMAPHORE.release()
        raise

def teardown_driver(driver):
    try:
        driver.quit()
    finally:
        DRIVER_SEMAPHORE.release()

def wait_for_window_app(driver, timeout=WAIT_WINDOW_APP):
    try:
        def _check(drv):
            try:
                val = drv.execute_script(
                    "return (typeof window !== 'undefined' && window.__APP) ? window.__APP : null;"
                )
                return val
            except JavascriptException:
                return False
        return WebDriverWait(driver, timeout).until(_check)
    except TimeoutException:
        return None

def get_nested_value(data_dict: dict, keys: list, default=None):
    dd = data_dict
    for key in keys:
        if isinstance(dd, list):
            try:
                dd = dd[key]
            except (IndexError, TypeError):
                return default
        else:
            dd = dd.get(key, default)
            if dd is default:
                return default
    return dd or default

def extract_id_from_url(url):
    m = re.search(r'-iid-(\d+)$', url)
    return m.group(1) if m else None

def parse_posting_date(raw):
    try:
        dt = dateparser.parse(raw)
        if dt:
            return dt.strftime("%d %B %Y"), dt.month, dt.year
    except Exception:
        return None, None, None
    return None, None, None

# PARSING DESKRIPSI RUMAH
def split_sentences(description):
    if not description:
        return []
    return [line.strip() for line in description.splitlines() if line.strip()]

def scrape_description(desc_lines, keywords, entity_type=None):
    for kata in keywords:
        if entity_type == 'electricity':
            pattern = rf'{kata}\s*[\:\.\-\s]*([\d.,]+)\s*(watt|va|kva|token|w|wt|kwh)?'
        elif entity_type in ['garage', 'carport']:
            pattern = rf'{kata}\s*[\:\.\-\s]*(\d+)?\s*(mobil|mbl|cars?)?'
        elif entity_type == 'heading':
            pattern = rf'{kata}\s*[\:\.\-\s]*(.*?)(Timur|Tenggara|Selatan|Barat Daya|Barat Laut|Barat|Utara|Timur Laut|Kiblat|Khiblat)'
        else:
            pattern = rf'{kata}\s*[\:\.\-\s]*([\w\s\.,]+)'

        for sentence in desc_lines:
            if kata.lower() in sentence.lower():
                match = re.search(pattern, sentence, re.IGNORECASE)
                if match:
                    if entity_type == 'electricity':
                        val = match.group(1).replace('.', '').replace(',', '')
                        return int(val)
                    elif entity_type in ['garage', 'carport']:
                        val = match.group(1)
                        return int(val) if val else 1
                    elif entity_type == 'heading':
                        h = match.group(2).capitalize()
                        if h.lower() in ['kiblat', 'khiblat']:
                            return 'Barat'
                        return h
                    else:
                        return match.group(1).strip()
    return None

def count_rooms(desc_lines, keywords):
    for kw in keywords:
        pattern = rf'(\d+)?\s*{kw}'
        for sentence in desc_lines:
            m = re.search(pattern, sentence, re.IGNORECASE)
            if m:
                return int(m.group(1)) if m.group(1) else 1
    return 0

def extract_detail_from_description(desc_lines):
    keywords = {
        'garage': ['Garasi', 'Garage'],
        'carport': ['Carport', 'Carpot'],
        'electricity': ['Listrik', 'electricity', 'pln', 'listtrik'],
        'heading': ['Hadap', 'Orientasi'],
        'ruang_tamu': ['Ruang Tamu'],
        'ruang_makan': ['Ruang Makan'],
        'maid_bedroom': ['Kamar Pembantu'],
        'maid_bathroom': ['Kamar Mandi Pembantu'],
        'floors': ['Lantai'],
    }
    info = {}
    info['garage'] = scrape_description(desc_lines, keywords['garage'], 'garage') or 0
    info['carport'] = scrape_description(desc_lines, keywords['carport'], 'carport') or 0
    info['electricity'] = scrape_description(desc_lines, keywords['electricity'], 'electricity')
    info['heading'] = scrape_description(desc_lines, keywords['heading'], 'heading')
    info['ruang_tamu'] = count_rooms(desc_lines, keywords['ruang_tamu'])
    info['ruang_makan'] = count_rooms(desc_lines, keywords['ruang_makan'])
    info['maid_bedroom'] = count_rooms(desc_lines, keywords['maid_bedroom'])
    info['maid_bathroom'] = count_rooms(desc_lines, keywords['maid_bathroom'])

    # jumlah lantai
    floor_val = None
    for sentence in desc_lines:
        match = re.search(r'(\d+)\s*lantai|lantai\s*[\:\-\s]*(\d+)', sentence, re.IGNORECASE)
        if match:
            floor_val = match.group(1) or match.group(2)
            break
    info['floors'] = int(floor_val) if floor_val else 1

    # ruang lain
    other_rooms = [s for s in desc_lines if "ruang" in s.lower() and "ruang tamu" not in s.lower() and "ruang makan" not in s.lower()]
    info['additional_rooms'] = 1 if other_rooms else 0
    return info

# SCRAPER PER URL
def scrape_single_url(url: str, headless=True):
    if not check_robots("https://www.olx.co.id"):
        return None

    ua = random.choice(USER_AGENTS)
    for attempt in range(1, MAX_RETRIES + 1):
        driver = None
        try:
            driver = setup_driver(ua, headless)
            driver.get(url)
            respectful_pause()

            app_data = wait_for_window_app(driver)
            if not app_data:
                scripts = driver.find_elements(By.TAG_NAME, "script")
                for s in scripts:
                    try:
                        text = s.get_attribute("innerHTML")
                        if text and "window.__APP" in text:
                            driver.execute_script(text)
                            app_data = wait_for_window_app(driver, timeout=5)
                            if app_data:
                                break
                    except StaleElementReferenceException:
                        continue

            if not app_data:
                raise RuntimeError("window.__APP not found")

            ads_id = extract_id_from_url(url)
            title = get_nested_value(app_data, ["states", "items", "elements", ads_id, "title"])
            price = get_nested_value(app_data, ["states", "items", "elements", ads_id, "price", "value", "raw"])
            ads_type = get_nested_value(app_data, ["states", "items", "elements", ads_id, "parameters"], [])
            desc = get_nested_value(app_data, ["states", "items", "elements", ads_id, "description"]) or ""
            desc_lines = split_sentences(desc)

            # lokasi dan parameter umum
            address_city = get_nested_value(app_data, ["states", "items", "elements", ads_id, "metadata", "locations", 0, "tree", 0, "addressComponents", 1, "name"])
            certificate = None
            params = get_nested_value(app_data, ["states", "items", "elements", ads_id, "parameters"], [])
            for p in params:
                if p.get("key_name", "").lower() == "sertifikasi":
                    val = p.get("value_name", "")
                    if "shm" in val.lower():
                        certificate = "SHM"
                    else:
                        certificate = val

            # detail deskripsi
            detail = extract_detail_from_description(desc_lines)

            posting_raw = get_nested_value(app_data, ["states", "items", "elements", ads_id, "created_at"])
            posting_date, posting_month, posting_year = parse_posting_date(posting_raw)

            result = {
                "url": url,
                "ads_id": ads_id,
                "title": title,
                "price": price,
                "city": address_city,
                "certificate": certificate,
                "garage_capacity": detail['garage'],
                "carport_capacity": detail['carport'],
                "electricity_capacity": detail['electricity'],
                "house_orientation": detail['heading'],
                "maid_bedrooms": detail['maid_bedroom'],
                "maid_bathrooms": detail['maid_bathroom'],
                "ruang_tamu": detail['ruang_tamu'],
                "ruang_makan": detail['ruang_makan'],
                "additional_rooms": detail['additional_rooms'],
                "floors": detail['floors'],
                "description": desc_lines,
                "posting_date": posting_date,
                "posting_month": posting_month,
                "posting_year": posting_year,
                "_scraped_at": datetime.utcnow().isoformat()
            }

            return result

        except Exception as e:
            logger.warning("Attempt %d failed for %s: %s", attempt, url, e)
            time.sleep((2 ** (attempt - 1)) + random.random())
        finally:
            if driver:
                teardown_driver(driver)
    return None

# CSV WRITER THREAD
def csv_writer_worker(fieldnames):
    with open(CSV_OUTPUT, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        while True:
            item = write_queue.get()
            if item is None:
                break
            if isinstance(item.get("description"), list):
                item["description"] = " ||| ".join(item["description"])
            writer.writerow(item)
            write_queue.task_done()

# MULTI-THREADING ORCHESTRATION
def scrape_urls_parallel(urls, max_workers=MAX_THREADS, headless=True):
    fieldnames = [
        "url", "ads_id", "title", "price", "city", "certificate",
        "garage_capacity", "carport_capacity", "electricity_capacity", "house_orientation",
        "maid_bedrooms", "maid_bathrooms", "ruang_tamu", "ruang_makan", "additional_rooms", "floors",
        "description", "posting_date", "posting_month", "posting_year", "_scraped_at"
    ]
    writer_thread = threading.Thread(target=csv_writer_worker, args=(fieldnames,), daemon=True)
    writer_thread.start()

    results = []
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        futures = {ex.submit(scrape_single_url, url, headless): url for url in urls}
        for fut in tqdm(as_completed(futures), total=len(futures), desc="Scraping URLs"):
            url = futures[fut]
            try:
                data = fut.result()
                if data:
                    write_queue.put(data)
                    results.append(data)
            except Exception as e:
                logger.error("Unhandled exception scraping %s: %s", url, e)

    write_queue.put(None)
    writer_thread.join()
    return results

# ENTRY POINT
def load_urls_from_csv(path="batch4.csv", url_col_index=4):
    urls = []
    with open(path, newline="", encoding="utf-8") as f:
        r = csv.reader(f)
        for row in r:
            if len(row) > url_col_index:
                urls.append(row[url_col_index])
            elif row:
                urls.append(row[-1])
    return urls

def main():
    urls = load_urls_from_csv("batch4.csv", url_col_index=4)
    logger.info("Starting scraping of %d URLs", len(urls))
    results = scrape_urls_parallel(urls, max_workers=MAX_THREADS, headless=True)
    logger.info("Finished scraping. Total success: %d", len(results))

if __name__ == "__main__":
    main()